# 📓 Chapter 3 — Practice Notebook
## Looking Inside Large Language Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---
### What you will practice
1. Inspect a real model's architecture and dimensions
2. Trace the full forward pass step by step
3. Observe how decoding strategy changes output
4. Measure the KV cache speedup
5. Visualize attention patterns
6. Knowledge check
---

In [ ]:
!pip install transformers accelerate -q
print('✅ Setup complete!')

---
## Part 1 — Load the Model and Inspect Its Architecture

Before we run anything, let's understand what we're working with.
The model architecture tells us everything about its capacity and design choices.

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_id = 'microsoft/Phi-3-mini-4k-instruct'

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(model_id)

print('Loading model...')
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map='auto',
    torch_dtype='auto',
    trust_remote_code=True,
)
model.eval()
print('✅ Model ready!')

In [ ]:
# ── DEMO: Print full architecture ─────────────────────────────────────────────
print(model)
print()
print('Key dimensions:')
print(f'  Vocabulary size:        {model.config.vocab_size:,}')
print(f'  Embedding dimensions:   {model.config.hidden_size:,}')
print(f'  Transformer blocks:     {model.config.num_hidden_layers}')
print(f'  Attention heads:        {model.config.num_attention_heads}')
print(f'  KV heads (GQA):         {model.config.num_key_value_heads}')
print(f'  Max context length:     {model.config.max_position_embeddings:,}')

In [ ]:
# ── DEMO: Count total parameters ─────────────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters:     {total_params:,}  ({total_params/1e9:.2f}B)')
print(f'Trainable parameters: {trainable_params:,}')
print()

# Break down by component
embed_params = sum(p.numel() for p in model.model.embed_tokens.parameters())
block_params = sum(p.numel() for p in model.model.layers.parameters())
head_params = sum(p.numel() for p in model.lm_head.parameters())

print('Parameter breakdown:')
print(f'  Embedding matrix:    {embed_params:>12,}  ({embed_params/total_params*100:.1f}%)')
print(f'  Transformer blocks:  {block_params:>12,}  ({block_params/total_params*100:.1f}%)')
print(f'  LM Head:             {head_params:>12,}  ({head_params/total_params*100:.1f}%)')

In [ ]:
# 🏋️ EXERCISE 1 — Grouped-Query Attention Check
# ─────────────────────────────────────────────────────────────────────────────
# Check if Phi-3 uses grouped-query attention.
# GQA = num_attention_heads > num_key_value_heads
#
# Tasks:
#   1. Print num_attention_heads and num_key_value_heads
#   2. How many heads share each K/V pair?
#   3. What type of attention is this: multi-head, multi-query, or grouped-query?

# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION
n_heads = model.config.num_attention_heads
n_kv = model.config.num_key_value_heads

print(f'Attention heads (Q):  {n_heads}')
print(f'KV heads:             {n_kv}')
print(f'Heads per KV group:   {n_heads // n_kv}')
print()
if n_kv == n_heads:
    print('Type: Multi-Head Attention (every head has own K, V)')
elif n_kv == 1:
    print('Type: Multi-Query Attention (all heads share one K, V)')
else:
    print(f'Type: Grouped-Query Attention ({n_heads} Q heads, {n_kv} K/V groups)')
    print(f'      Each group of {n_heads//n_kv} Q heads shares one K, V pair')

print()
print('💡 GQA reduces KV cache size by', n_heads // n_kv, 'x compared to multi-head.')
print('   This means faster inference and lower VRAM usage.')

---
## Part 2 — Tracing the Forward Pass

Let's manually trace each step of a single forward pass — from text to the predicted next token.
This is what happens inside every API call you make to an LLM.

In [ ]:
# ── DEMO: Step-by-step forward pass ───────────────────────────────────────────
prompt = 'The capital of France is'

# Step 1: Text → Token IDs
input_ids = tokenizer(prompt, return_tensors='pt').input_ids
device = next(model.parameters()).device
input_ids = input_ids.to(device)

print('STEP 1 — Tokenization')
print(f'  Prompt: "{prompt}"')
print(f'  Token IDs: {input_ids[0].tolist()}')
print(f'  Tokens: {[tokenizer.decode(t) for t in input_ids[0]]}')
print(f'  Shape: {input_ids.shape}  [batch_size, num_tokens]')

In [ ]:
# Step 2-3: Token IDs → Transformer blocks → output vectors
with torch.no_grad():
    model_output = model.model(input_ids)

hidden_states = model_output[0]

print('STEP 2-3 — Through all Transformer blocks')
print(f'  Output shape: {hidden_states.shape}')
print(f'  = [batch={hidden_states.shape[0]}, tokens={hidden_states.shape[1]}, dims={hidden_states.shape[2]}]')
print(f'  Each of the {hidden_states.shape[1]} tokens now has a context-aware vector')
print(f'  We only need the LAST token\'s vector for prediction')
print(f'  Last token vector (first 5 values): {hidden_states[0, -1, :5].tolist()}')

In [ ]:
# Step 4: LM Head → probability scores
with torch.no_grad():
    lm_head_output = model.lm_head(hidden_states)

print('STEP 4 — LM Head')
print(f'  Input shape:  {hidden_states.shape}')
print(f'  Output shape: {lm_head_output.shape}')
print(f'  = {lm_head_output.shape[2]:,} scores (one per vocab token)')
print()

# Get scores for last token position
import torch.nn.functional as F
last_token_scores = lm_head_output[0, -1]  # [32064]
probabilities = F.softmax(last_token_scores, dim=-1)

# Top 10 candidates
top_probs, top_ids = torch.topk(probabilities, 10)

print('Top 10 candidate next tokens:')
print(f'  {"Token":<15} {"Probability"}')
print('  ' + '-' * 30)
for prob, tid in zip(top_probs, top_ids):
    token_text = tokenizer.decode(tid)
    bar = '█' * int(prob.item() * 50)
    print(f'  \'{token_text}\'\t{prob.item():.4f}  {bar}')

In [ ]:
# Step 5: Pick the next token
next_token_id = last_token_scores.argmax(-1)
next_token_text = tokenizer.decode(next_token_id)

print('STEP 5 — Token Selection (Greedy)')
print(f'  Next token ID:   {next_token_id.item()}')
print(f'  Next token text: \'{next_token_text}\'')
print(f'  Full output so far: "{prompt}{next_token_text}"')

In [ ]:
# 🏋️ EXERCISE 2 — Run the Loop Manually
# ─────────────────────────────────────────────────────────────────────────────
# Autoregressive generation = forward pass in a loop.
# Implement this loop manually to generate 5 tokens.
#
# Tasks:
#   1. Start with prompt = "Machine learning is"
#   2. Run a forward pass → get next token → append to input
#   3. Repeat 5 times
#   4. Print the generated text after each step
#
# Hint: you already know how to do a forward pass from the demo above

start_prompt = 'Machine learning is'

# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION

current_ids = tokenizer(start_prompt, return_tensors='pt').input_ids.to(device)
print(f'Start: "{start_prompt}"')
print()

for step in range(5):
    with torch.no_grad():
        model_out = model.model(current_ids)
        lm_out = model.lm_head(model_out[0])
    
    # Greedy: pick highest probability token
    next_id = lm_out[0, -1].argmax(-1).unsqueeze(0).unsqueeze(0)
    next_text = tokenizer.decode(next_id[0][0])
    
    # Append to input
    current_ids = torch.cat([current_ids, next_id], dim=1)
    
    current_text = tokenizer.decode(current_ids[0])
    print(f'Step {step+1}: added \'{next_text}\'')
    print(f'  Text: "{current_text}"')
    print()

print('💡 This is EXACTLY what model.generate() does under the hood.')
print('   It just adds optimizations like KV cache and stopping criteria.')

---
## Part 3 — Decoding Strategies and Temperature

The same model produces very different outputs depending on your decoding strategy.
This is one of the most practical things to understand for building LLM apps.

In [ ]:
# ── Setup pipeline for decoding experiments ───────────────────────────────────
from transformers import pipeline

base_pipeline = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=60,
)

prompt_messages = [{'role': 'user', 'content': 'Write a one-sentence description of the ocean.'}]

def generate(messages, **kwargs):
    out = base_pipeline(messages, **kwargs)
    return out[0]['generated_text'].strip()

In [ ]:
# ── DEMO: Same prompt, different temperatures ─────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

print('GREEDY (temperature=0, deterministic):')
for i in range(3):
    print(f'  Run {i+1}: {generate(prompt_messages, do_sample=False)}')
print()

print('SAMPLING temperature=0.7 (balanced):')
for i in range(3):
    print(f'  Run {i+1}: {generate(prompt_messages, do_sample=True, temperature=0.7)}')
print()

print('SAMPLING temperature=1.5 (creative/chaotic):')
for i in range(3):
    print(f'  Run {i+1}: {generate(prompt_messages, do_sample=True, temperature=1.5)}')

In [ ]:
# 🏋️ EXERCISE 3 — Pick the Right Temperature for Each Use Case
# ─────────────────────────────────────────────────────────────────────────────
# For each use case, pick an appropriate temperature and run it.
# Observe how the outputs change.

use_cases = [
    {
        'name': 'Factual Q&A',
        'prompt': [{'role': 'user', 'content': 'What is the boiling point of water in Celsius?'}],
        'your_temperature': 0.0,  # CHANGE THIS — what makes sense here?
    },
    {
        'name': 'Code generation',
        'prompt': [{'role': 'user', 'content': 'Write a Python function to check if a number is prime.'}],
        'your_temperature': 0.0,  # CHANGE THIS
    },
    {
        'name': 'Creative story opening',
        'prompt': [{'role': 'user', 'content': 'Write the opening sentence of a mystery novel.'}],
        'your_temperature': 0.0,  # CHANGE THIS
    },
]

for uc in use_cases:
    temp = uc['your_temperature']
    do_sample = temp > 0
    result = generate(uc['prompt'], do_sample=do_sample, temperature=temp if do_sample else None)
    print(f'Use case: {uc["name"]} (temp={temp})')
    print(f'Output: {result}')
    print()

---
## Part 4 — KV Cache Speedup

The KV cache is one of the most important optimizations in LLM inference.
Let's measure the actual performance difference.

In [ ]:
# ── DEMO: Measure KV cache speedup ───────────────────────────────────────────
import time

long_prompt = 'Write a very long email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.'
input_ids = tokenizer(long_prompt, return_tensors='pt').input_ids.to(device)

def time_generation(use_cache, num_tokens=100, runs=3):
    times = []
    for _ in range(runs):
        start = time.time()
        with torch.no_grad():
            model.generate(
                input_ids,
                max_new_tokens=num_tokens,
                use_cache=use_cache,
                do_sample=False,
            )
        times.append(time.time() - start)
    return sum(times) / len(times)

print(f'Generating {100} tokens...')
print()

print('With KV cache (use_cache=True):')
t_with = time_generation(use_cache=True)
print(f'  Average: {t_with:.2f} seconds')
print()

print('Without KV cache (use_cache=False):')
t_without = time_generation(use_cache=False)
print(f'  Average: {t_without:.2f} seconds')
print()

print(f'Speedup: {t_without/t_with:.1f}x faster with cache')
print()
print('💡 Always keep use_cache=True in production. This is the default.')

In [ ]:
# ── DEMO: KV cache memory cost ────────────────────────────────────────────────
# The KV cache grows with context length — this is why long contexts use more VRAM

def estimate_kv_cache_mb(model_config, num_tokens, dtype_bytes=2):
    """Estimate KV cache memory usage in MB."""
    # Each token: 2 (K and V) × num_layers × num_kv_heads × head_dim × dtype_bytes
    head_dim = model_config.hidden_size // model_config.num_attention_heads
    kv_per_token = 2 * model_config.num_hidden_layers * model_config.num_key_value_heads * head_dim * dtype_bytes
    total_bytes = kv_per_token * num_tokens
    return total_bytes / (1024 ** 2)

print('KV Cache memory usage for Phi-3-mini:')
print(f'{"Context tokens":<20} {"KV Cache (MB)"}')
print('-' * 35)
for n_tokens in [512, 1024, 2048, 4096, 8192, 32768, 131072]:
    mb = estimate_kv_cache_mb(model.config, n_tokens)
    bar = '█' * min(int(mb / 20), 30)
    print(f'{n_tokens:<20,} {mb:>8.1f} MB  {bar}')

print()
print('💡 This is why GPT-4\'s 128K context window requires massive VRAM.')
print('   Each token you send in your prompt costs cache memory for the full session.')

---
## Part 5 — Streaming Tokens

In real applications, you don't wait for the entire response before showing it to the user.
You stream tokens as they are generated — one by one.
This is what creates the 'typing' effect in ChatGPT.

In [ ]:
# ── DEMO: Streaming token generation ─────────────────────────────────────────
from transformers import TextStreamer
import sys

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

messages = [{'role': 'user', 'content': 'Explain what an API is in 3 sentences.'}]
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors='pt').to(device)

print('Streaming output (tokens appear as they are generated):')
print('-' * 60)
with torch.no_grad():
    model.generate(
        **inputs,
        max_new_tokens=100,
        streamer=streamer,
        do_sample=False,
    )
print('\n' + '-' * 60)
print('💡 Each token is printed as soon as the model generates it.')
print('   This is how all production LLM APIs work (streaming=True).')

---
## Part 6 — Knowledge Check

In [ ]:
# ❓ Q1: What does 'autoregressive' mean in the context of LLM generation?
#        Why does this mean the model generates one token at a time?
your_answer = """
...
"""
print(your_answer)

In [ ]:
# ❓ Q2: For Phi-3-mini, the LM Head is Linear(3072 → 32064).
#        What do the numbers 3072 and 32064 represent?
your_answer = """
...
"""
print(your_answer)

In [ ]:
# ❓ Q3: PRACTICAL SCENARIO
#   Your chatbot app is responding slowly. A user types a message, and waits
#   5 seconds before seeing any response. The generation itself takes 4 seconds.
#
#   What are TWO strategies to improve the user experience — without
#   making the model actually faster?
your_answer = """
...
"""
print(your_answer)

In [ ]:
# ❓ Q4: Explain the Query, Key, Value mechanism in attention using
#        a developer-friendly analogy of your own (not the database one from notes).
your_answer = """
...
"""
print(your_answer)

In [ ]:
# ✅ MODEL ANSWERS

answers = {
    'Q1': '''Autoregressive means the model uses its own previously generated output
as input for the next prediction. Each new token depends on all previous tokens.
Since each token depends on the previous one, they must be generated sequentially —
you can't generate token 5 until token 4 exists, because token 4 is part of the input.
This is fundamentally different from, say, an image classifier that processes
all pixels at once.''',

    'Q2': '''3072 = the model's hidden dimension (embedding dimension).
Every token is represented as a vector of 3,072 numbers throughout the model.
This is the output of the last Transformer block for the final token position.

32064 = the vocabulary size (number of unique tokens the model knows).
The LM Head outputs one probability score for each of the 32,064 possible
next tokens. The token with the highest score (or sampled from the distribution)
becomes the next generated token.''',

    'Q3': '''Strategy 1: STREAMING
Send tokens to the user as they are generated instead of waiting for the full
response. The user sees the first token within ~200ms instead of waiting 4 seconds.
Implemented with TextStreamer in Transformers or streaming=True in OpenAI API.

Strategy 2: SHOW A LOADING STATE / TYPING INDICATOR
While generating, show a typing animation or progress indicator so the user knows
something is happening. Even without streaming, this significantly reduces perceived
wait time because the user isn't staring at a blank screen wondering if it crashed.

Bonus: Pre-generation triggers — start generating as soon as the user starts typing
(predict likely requests) to reduce latency for common queries.''',

    'Q4': '''Many valid analogies — one example:
Think of attention like a job posting system:
- Query = a recruiter's job requirements (what skills are needed)
- Keys = each candidate's resume summary (what they offer)
- Values = each candidate's full profile (their actual knowledge)
- Attention score = how well the candidate matches the job requirements
- Output = a blended "ideal candidate" weighted by match score

When processing "it" in "Sarah fed the cat because it",
the Query asks "who is this pronoun referring to?"
The Key for "cat" scores HIGH (it's a noun that could be referred to)
The model pulls in the Value (cat information) to resolve the pronoun.'''
}

for q, ans in answers.items():
    print('=' * 65)
    print(f'✅ {q}:')
    print(ans)
    print()

---
## 🎉 Chapter 3 Practice Complete!

### What you did:
- ✅ Inspected a real model's architecture (vocab size, dims, number of blocks)
- ✅ Counted and broke down parameters by component
- ✅ Confirmed Phi-3 uses Grouped-Query Attention
- ✅ Manually traced a full forward pass (tokenize → blocks → LM Head → decode)
- ✅ Implemented autoregressive generation in a manual loop
- ✅ Experimented with temperature and decoding strategies
- ✅ Measured KV cache speedup (~5x)
- ✅ Estimated KV cache memory cost at different context lengths
- ✅ Implemented token streaming
- ✅ Answered all knowledge check questions

### Before Chapter 4:
Read `summary.md` — 10 minutes, locks in the entire chapter.

### What's next:
**Chapter 4 — Text Classification**
The internals are done. From now on it's all about APPLICATIONS.
Chapter 4 covers how to classify text (sentiment analysis, spam detection, topic labeling)
using both encoder-only models (BERT) and generative models (GPT).